# Phase 9 — Final visualizations

This notebook creates publication-quality visual summaries from the validated
outputs of Phases 1–8. It does **not** preprocess cells, recluster data, retrain
classifiers, search literature, or alter biological interpretations.

Each figure section names its source files. Missing inputs are handled
independently: the affected figure is marked `SKIPPED` or `FAILED` in the
manifest while the notebook continues.

## Centralized configuration and helpers

In [1]:
import os
import json
import textwrap
import warnings
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/pbmc3k_phase9_matplotlib")

import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch, Patch
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy import sparse

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Locate PBMC3k from the repository root, PBMC3k/, or PBMC3k/notebooks/.
cwd = Path.cwd().resolve()
candidates = []
for base in (cwd, *cwd.parents):
    candidates.extend((base, base / "PBMC3k", base / "26-the-backpropagators-analysis" / "PBMC3k"))
PROJECT = next(
    (p for p in candidates if (p / "results" / "phase8" / "all_clusters_summary.csv").is_file()),
    None,
)
if PROJECT is None:
    raise FileNotFoundError(
        "PBMC3k project root not found. Expected results/phase8/all_clusters_summary.csv "
        "beneath the current directory or one of its parents."
    )

OUTPUT_DIR = PROJECT / "results" / "phase9"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PNG_DPI = 320
DEFAULT_FIGSIZE = (10, 6)
EXPORT_FORMATS = ("png", "svg", "pdf")
CLUSTER_ORDER = [str(i) for i in range(9)]

mpl.rcParams.update({
    "figure.figsize": DEFAULT_FIGSIZE,
    "figure.dpi": 120,
    "savefig.dpi": PNG_DPI,
    "savefig.facecolor": "white",
    "axes.facecolor": "white",
    "figure.facecolor": "white",
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
sns.set_theme(style="whitegrid", context="paper", rc=mpl.rcParams)

SOURCE = {
    "adata": PROJECT / "data" / "processed" / "pbmc3k_phase3_annotated.h5ad",
    "annotations": PROJECT / "results" / "tables" / "leiden_9_cell_type_annotations.csv",
    "markers": PROJECT / "results" / "phase6" / "selected_marker_genes.csv",
    "models": PROJECT / "results" / "tables" / "classification_model_comparison.csv",
    "phase7_literature": PROJECT / "results" / "phase7" / "literature_summary.csv",
    "phase7_references": PROJECT / "results" / "phase7" / "references.csv",
    "phase7_coverage": PROJECT / "results" / "phase7" / "phase7_coverage_summary.csv",
    "phase7_reuse": PROJECT / "results" / "phase7" / "evidence_reuse_report.csv",
    "phase7_validation": PROJECT / "results" / "phase7" / "phase7_validation_report.json",
    "phase8_summary": PROJECT / "results" / "phase8" / "all_clusters_summary.csv",
}

manifest_rows = []

def rel(path):
    path = Path(path)
    try:
        return str(path.relative_to(PROJECT))
    except ValueError:
        return str(path)

def require_files(paths):
    missing = [Path(p) for p in paths if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("Missing required source file(s): " + ", ".join(rel(p) for p in missing))

def save_figure(fig, stem, formats=("png",), **kwargs):
    outputs = []
    for extension in formats:
        path = OUTPUT_DIR / f"{stem}.{extension}"
        save_kwargs = {"bbox_inches": "tight", "facecolor": "white"}
        if extension == "png":
            save_kwargs["dpi"] = PNG_DPI
        save_kwargs.update(kwargs)
        fig.savefig(path, **save_kwargs)
        outputs.append(path)
    plt.close(fig)
    return outputs

def record(number, name, sources, outputs, status, notes=""):
    row = {
        "figure_number": number,
        "figure_name": name,
        "source_files": "; ".join(rel(p) for p in sources),
        "output_files": "; ".join(rel(p) for p in outputs),
        "status": status,
        "notes": notes,
    }
    manifest_rows.append(row)
    print(f"Figure {number}: {status} — {name}")
    if notes:
        print(" ", notes)

def cluster_sort_key(values):
    return pd.to_numeric(values, errors="coerce")

print("Project:", PROJECT)
print("Output:", OUTPUT_DIR)
print("Random seed:", RANDOM_SEED)

Project: /Users/shelyjain/Desktop/Desktop - Shely's Macbook Pro/cosmos/26-the-backpropagators-analysis/PBMC3k
Output: /Users/shelyjain/Desktop/Desktop - Shely's Macbook Pro/cosmos/26-the-backpropagators-analysis/PBMC3k/results/phase9
Random seed: 42


In [2]:
# Load the small validated tables used across figures. Individual figure
# sections still validate their own required sources.
annotations = None
markers = None
phase8 = None
adata = None

if SOURCE["annotations"].is_file():
    annotations = pd.read_csv(SOURCE["annotations"], dtype={"leiden": str})
if SOURCE["markers"].is_file():
    markers = pd.read_csv(SOURCE["markers"], dtype={"cluster": str})
if SOURCE["phase8_summary"].is_file():
    phase8 = pd.read_csv(SOURCE["phase8_summary"], dtype={"Cluster ID": str})
if SOURCE["adata"].is_file():
    adata = sc.read_h5ad(SOURCE["adata"])

# Derive the categorical palette from the validated cluster order.
if annotations is not None:
    annotations = annotations.sort_values("leiden", key=cluster_sort_key).reset_index(drop=True)
    ordered_cell_types = annotations["cell_type"].tolist()
else:
    ordered_cell_types = []
palette = sns.color_palette("colorblind", n_colors=max(9, len(ordered_cell_types)))
CELL_TYPE_COLORS = {
    cell_type: mpl.colors.to_hex(palette[i])
    for i, cell_type in enumerate(ordered_cell_types)
}
(OUTPUT_DIR / "cell_type_color_mapping.json").write_text(
    json.dumps(CELL_TYPE_COLORS, indent=2) + "\n", encoding="utf-8"
)

print("Cluster order:", CLUSTER_ORDER)
print("Cell-type colors:")
print(json.dumps(CELL_TYPE_COLORS, indent=2))

Cluster order: ['0', '1', '2', '3', '4', '5', '6', '7', '8']
Cell-type colors:
{
  "Cytotoxic CD8 T cells": "#0173b2",
  "B cells": "#de8f05",
  "IL7R+ memory/helper T cells": "#029e73",
  "Classical monocytes": "#d55e00",
  "CD16+ non-classical monocytes": "#cc78bc",
  "NK cells": "#ca9161",
  "Activated/transitional T cells": "#fbafe4",
  "Naive/resting T cells": "#949494",
  "Platelets": "#ece133"
}


## Figure 1 — Final annotated UMAP

Sources: `data/processed/pbmc3k_phase3_annotated.h5ad` and
`results/tables/leiden_9_cell_type_annotations.csv`.

In [3]:
sources = [SOURCE["adata"], SOURCE["annotations"]]
outputs = []
try:
    require_files(sources)
    if adata is None or "X_umap" not in adata.obsm:
        raise ValueError("The annotated AnnData object does not contain obsm['X_umap'].")
    required_obs = {"leiden", "cell_type"}
    if not required_obs.issubset(adata.obs.columns):
        raise ValueError(f"AnnData is missing required obs columns: {sorted(required_obs - set(adata.obs.columns))}")

    expected = annotations.set_index("leiden")["cell_type"].to_dict()
    observed_pairs = (
        adata.obs.assign(leiden_string=adata.obs["leiden"].astype(str))
        .groupby("leiden_string", observed=True)["cell_type"]
        .agg(lambda x: sorted(set(x.astype(str))))
        .to_dict()
    )
    if any(observed_pairs.get(cluster) != [cell_type] for cluster, cell_type in expected.items()):
        raise ValueError("Cluster-to-cell-type labels in AnnData disagree with the validated annotation table.")

    xy = np.asarray(adata.obsm["X_umap"])
    fig, ax = plt.subplots(figsize=(11.5, 7.5))
    for cluster in CLUSTER_ORDER:
        cell_type = expected[cluster]
        mask = adata.obs["leiden"].astype(str).to_numpy() == cluster
        ax.scatter(
            xy[mask, 0], xy[mask, 1], s=9, alpha=0.82, linewidths=0,
            color=CELL_TYPE_COLORS[cell_type], label=cell_type, rasterized=True,
        )
        center = np.median(xy[mask], axis=0)
        ax.text(
            center[0], center[1], cluster, ha="center", va="center",
            fontsize=9, fontweight="bold",
            bbox={"boxstyle": "circle,pad=0.25", "facecolor": "white",
                  "edgecolor": CELL_TYPE_COLORS[cell_type], "linewidth": 1.4, "alpha": 0.92},
        )
    ax.set(title="PBMC3K final validated cell-type annotations", xlabel="UMAP 1", ylabel="UMAP 2")
    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(
        title="Final cell type", bbox_to_anchor=(1.02, 0.5), loc="center left",
        frameon=False, markerscale=2, borderaxespad=0,
    )
    fig.tight_layout()
    outputs = save_figure(fig, "figure_1_annotated_umap", EXPORT_FORMATS)
    record(1, "Final annotated UMAP", sources, outputs, "COMPLETE",
           f"All {adata.n_obs:,} cells shown; numbered labels indicate Leiden clusters.")
except Exception as exc:
    record(1, "Final annotated UMAP", sources, outputs, "SKIPPED", str(exc))

Figure 1: SKIPPED — Final annotated UMAP
  unhashable type: 'list'


## Figure 2 — Cluster and cell-type composition

Sources: the final per-cell Leiden labels in the annotated AnnData object and
the validated annotation table.

In [4]:
sources = [SOURCE["adata"], SOURCE["annotations"]]
outputs = []
try:
    require_files(sources)
    if adata is None or not {"leiden", "cell_type"}.issubset(adata.obs.columns):
        raise ValueError("Annotated AnnData must contain obs['leiden'] and obs['cell_type'].")
    composition = (
        adata.obs.assign(
            cluster_id=adata.obs["leiden"].astype(str),
            cell_type_string=adata.obs["cell_type"].astype(str),
        )
        .groupby(["cluster_id", "cell_type_string"], observed=True)
        .size().rename("cell_count").reset_index()
        .rename(columns={"cell_type_string": "cell_type"})
    )
    composition["percentage_of_total"] = 100 * composition["cell_count"] / adata.n_obs
    composition = composition.sort_values("cluster_id", key=cluster_sort_key).reset_index(drop=True)
    if len(composition) != len(annotations):
        raise ValueError("Expected one final cell type for every cluster.")
    composition_path = OUTPUT_DIR / "cluster_composition.csv"
    composition.to_csv(composition_path, index=False)
    colors = [CELL_TYPE_COLORS[x] for x in composition["cell_type"]]
    labels = [f"{r.cluster_id} · {r.cell_type}" for r in composition.itertuples()]

    fig, ax = plt.subplots(figsize=(12.5, 6.5))
    bars = ax.bar(labels, composition["cell_count"], color=colors, edgecolor="white")
    ax.bar_label(bars, labels=[f"{x:,}" for x in composition["cell_count"]], padding=3, fontsize=9)
    ax.set(title="PBMC3K cluster composition — cell counts", xlabel="Cluster · final cell type", ylabel="Cells")
    ax.tick_params(axis="x", rotation=35)
    for tick in ax.get_xticklabels():
        tick.set_ha("right")
    ax.set_ylim(0, composition["cell_count"].max() * 1.16)
    fig.tight_layout()
    outputs += save_figure(fig, "figure_2_cluster_cell_counts", ("png",))

    fig, ax = plt.subplots(figsize=(12.5, 6.5))
    bars = ax.bar(labels, composition["percentage_of_total"], color=colors, edgecolor="white")
    ax.bar_label(
        bars, labels=[f"{x:.1f}%" for x in composition["percentage_of_total"]],
        padding=3, fontsize=9,
    )
    ax.set(title="PBMC3K cluster composition — percentage of all cells",
           xlabel="Cluster · final cell type", ylabel="Cells (%)")
    ax.tick_params(axis="x", rotation=35)
    for tick in ax.get_xticklabels():
        tick.set_ha("right")
    ax.set_ylim(0, composition["percentage_of_total"].max() * 1.18)
    fig.tight_layout()
    outputs += save_figure(fig, "figure_2_cluster_percentages", ("png",))
    outputs.append(composition_path)
    record(2, "Cluster and cell-type composition", sources, outputs, "COMPLETE",
           f"Counts and percentages sum to {composition['cell_count'].sum():,} cells and "
           f"{composition['percentage_of_total'].sum():.1f}%.")
except Exception as exc:
    record(2, "Cluster and cell-type composition", sources, outputs, "SKIPPED", str(exc))

Figure 2: COMPLETE — Cluster and cell-type composition
  Counts and percentages sum to 2,638 cells and 100.0%.


## Figure 3 — Representative marker-gene heatmap

Sources: `results/phase6/selected_marker_genes.csv` and the immutable `.raw`
normalized expression matrix in the annotated AnnData object.

The matrix uses cluster-average log-normalized expression. For visualization,
each gene is z-scored across the nine cluster averages:
`z = (cluster mean − gene mean) / gene standard deviation`. This transformation
is applied only to the derived plotting table and does not modify AnnData.

In [5]:
sources = [SOURCE["adata"], SOURCE["markers"]]
outputs = []
try:
    require_files(sources)
    if adata is None or adata.raw is None:
        raise ValueError("Annotated AnnData does not contain the required immutable raw expression matrix.")
    if markers is None:
        raise ValueError("Phase 6 selected marker table could not be loaded.")

    selected = (
        markers.sort_values(["cluster", "representative_rank"], key=lambda s: cluster_sort_key(s) if s.name == "cluster" else s)
        .drop_duplicates("gene", keep="first")
        .reset_index(drop=True)
    )
    genes = selected["gene"].astype(str).tolist()
    absent = sorted(set(genes) - set(adata.raw.var_names))
    if absent:
        raise ValueError(f"{len(absent)} representative genes are absent from AnnData.raw: {absent}")

    expression = adata.raw[:, genes].X
    cluster_means = []
    leiden_strings = adata.obs["leiden"].astype(str).to_numpy()
    for cluster in CLUSTER_ORDER:
        block = expression[leiden_strings == cluster]
        mean = np.asarray(block.mean(axis=0)).ravel() if sparse.issparse(block) else np.asarray(block).mean(axis=0)
        cluster_means.append(mean)
    mean_matrix = np.vstack(cluster_means)
    gene_mean = mean_matrix.mean(axis=0)
    gene_sd = mean_matrix.std(axis=0, ddof=0)
    gene_sd[gene_sd == 0] = 1.0
    z_matrix = (mean_matrix - gene_mean) / gene_sd

    cell_type_by_cluster = annotations.set_index("leiden")["cell_type"].to_dict()
    matrix_df = pd.DataFrame(z_matrix, columns=genes)
    matrix_df.insert(0, "cell_type", [cell_type_by_cluster[c] for c in CLUSTER_ORDER])
    matrix_df.insert(0, "cluster_id", CLUSTER_ORDER)
    matrix_path = OUTPUT_DIR / "marker_gene_heatmap_matrix.csv"
    matrix_df.to_csv(matrix_path, index=False)

    source_cluster = selected.set_index("gene")["cluster"].astype(str).to_dict()
    display_labels = [f"{gene}\n(C{source_cluster[gene]})" for gene in genes]
    row_labels = [f"C{c} · {cell_type_by_cluster[c]}" for c in CLUSTER_ORDER]
    fig, ax = plt.subplots(figsize=(25, 8.5))
    sns.heatmap(
        z_matrix, cmap="vlag", center=0, vmin=-2.3, vmax=2.3,
        xticklabels=display_labels, yticklabels=row_labels,
        linewidths=0.15, linecolor="white",
        cbar_kws={"label": "Gene-wise z-score of cluster-average expression", "shrink": 0.75},
        ax=ax,
    )
    ax.set(
        title=f"Phase 6 representative markers — {len(genes)} unique genes from 90 cluster-gene entries",
        xlabel="Representative gene (C# = Phase 6 source cluster)",
        ylabel="Final cluster · cell type",
    )
    ax.tick_params(axis="x", labelrotation=90, labelsize=7)
    ax.tick_params(axis="y", labelrotation=0)
    fig.tight_layout()
    outputs = save_figure(fig, "figure_3_marker_gene_heatmap", EXPORT_FORMATS)
    outputs.append(matrix_path)
    record(3, "Representative marker-gene heatmap", sources, outputs, "COMPLETE",
           f"Plotted {len(genes)} deduplicated Phase 6 genes as gene-wise z-scores of cluster averages.")
except Exception as exc:
    record(3, "Representative marker-gene heatmap", sources, outputs, "SKIPPED", str(exc))

Figure 3: COMPLETE — Representative marker-gene heatmap
  Plotted 78 deduplicated Phase 6 genes as gene-wise z-scores of cluster averages.


## Figure 4 — Marker specificity summary

Source: saved Phase 6 representative-marker statistics. No marker statistics
are recomputed.

In [6]:
sources = [SOURCE["markers"]]
outputs = []
try:
    require_files(sources)
    required = {
        "cluster", "cell_type", "representative_rank", "gene", "marker_score",
        "avg_log2FC", "specificity_delta", "adjusted_p_value", "pct_in", "pct_out",
    }
    if markers is None or not required.issubset(markers.columns):
        raise ValueError(f"Phase 6 marker table is missing columns: {sorted(required - set(markers.columns if markers is not None else []))}")
    summary = (
        markers.sort_values(["cluster", "representative_rank"], key=lambda s: cluster_sort_key(s) if s.name == "cluster" else s)
        .groupby("cluster", observed=True, sort=False).head(5).copy()
    )
    summary_path = OUTPUT_DIR / "marker_specificity_summary.csv"
    summary[list(required)].sort_values(["cluster", "representative_rank"]).to_csv(summary_path, index=False)

    fig, axes = plt.subplots(3, 3, figsize=(15, 13))
    for ax, cluster in zip(axes.flat, CLUSTER_ORDER):
        part = summary[summary["cluster"].eq(cluster)].sort_values("marker_score")
        cell_type = part["cell_type"].iloc[0]
        bars = ax.barh(part["gene"], part["marker_score"], color=CELL_TYPE_COLORS[cell_type], alpha=0.9)
        ax.bar_label(bars, labels=[f"{x:.1f}" for x in part["marker_score"]], padding=2, fontsize=7)
        ax.set_title(f"C{cluster} · {cell_type}", fontsize=10, fontweight="bold")
        ax.set_xlabel("Marker score")
        ax.grid(axis="y", visible=False)
        ax.set_xlim(0, summary["marker_score"].max() * 1.12)
    fig.suptitle("Strongest Phase 6 representative markers by cluster", fontsize=16, y=1.01)
    fig.tight_layout()
    outputs = save_figure(fig, "figure_4_marker_specificity", ("png",))
    outputs.append(summary_path)
    record(4, "Marker specificity summary", sources, outputs, "COMPLETE",
           "Top five saved representative markers per cluster; bar length is the stored marker score.")
except Exception as exc:
    record(4, "Marker specificity summary", sources, outputs, "SKIPPED", str(exc))

Figure 4: COMPLETE — Marker specificity summary
  Top five saved representative markers per cluster; bar length is the stored marker score.


## Figure 5 — Machine-learning model comparison

Source: `results/tables/classification_model_comparison.csv`. The comparison is
restricted to the three requested models. “Best” follows Phase 5's saved rank,
which was selected using validation macro-F1 (not the held-out test scores).
The only saved cross-validation uncertainty is the standard deviation of
training CV macro-F1, so uncertainty bars are shown only for that metric.

In [7]:
sources = [SOURCE["models"]]
outputs = []
try:
    require_files(sources)
    models = pd.read_csv(SOURCE["models"])
    canonical = {
        "Logistic regression": "Logistic Regression",
        "Random forest": "Random Forest",
        "XGBoost": "XGBoost",
    }
    selected_models = models[models["model"].isin(canonical)].copy()
    if set(selected_models["model"]) != set(canonical):
        raise ValueError("One or more requested classifiers are absent from the saved Phase 5 table.")
    selected_models["model_display"] = selected_models["model"].map(canonical)
    selected_models = selected_models.sort_values("rank")

    metrics = [
        ("validation_accuracy", "Accuracy"),
        ("validation_macro_precision", "Macro precision"),
        ("validation_macro_recall", "Macro recall"),
        ("validation_macro_f1", "Macro F1"),
        ("validation_roc_auc_ovr_macro", "ROC-AUC"),
        ("training_cv_macro_f1_mean", "CV macro F1"),
    ]
    long_rows = []
    for row in selected_models.itertuples(index=False):
        for column, metric_label in metrics:
            long_rows.append({
                "model": row.model_display,
                "phase5_rank": int(row.rank),
                "metric": metric_label,
                "value": float(getattr(row, column)),
                "uncertainty_std": (
                    float(row.training_cv_macro_f1_std) if metric_label == "CV macro F1" else np.nan
                ),
                "selection_partition": "training CV" if metric_label == "CV macro F1" else "validation",
                "is_phase5_selected_best": int(row.rank) == 1,
            })
    comparison_summary = pd.DataFrame(long_rows)
    summary_path = OUTPUT_DIR / "model_comparison_summary.csv"
    comparison_summary.to_csv(summary_path, index=False)

    metric_names = [metric_label for _, metric_label in metrics]
    x = np.arange(len(metric_names))
    width = 0.24
    fig, ax = plt.subplots(figsize=(15, 7))
    model_colors = dict(zip(canonical.values(), sns.color_palette("colorblind", 3)))
    for i, row in enumerate(selected_models.itertuples(index=False)):
        values = [float(getattr(row, column)) for column, _ in metrics]
        errors = [0, 0, 0, 0, 0, float(row.training_cv_macro_f1_std)]
        label = row.model_display + (" [Phase 5 selected]" if int(row.rank) == 1 else "")
        bars = ax.bar(
            x + (i - 1) * width, values, width, label=label,
            color=model_colors[row.model_display], yerr=errors, capsize=3,
            edgecolor="#333333" if int(row.rank) == 1 else "white",
            linewidth=1.4 if int(row.rank) == 1 else 0.5,
        )
    ax.set(
        title="Phase 5 classifier comparison — saved validation metrics",
        xlabel="Metric", ylabel="Score",
        xticks=x, xticklabels=metric_names, ylim=(0.72, 1.02),
    )
    ax.axhline(1.0, color="#666666", linewidth=0.7)
    ax.legend(bbox_to_anchor=(1.02, 0.62), loc="center left", frameon=True, ncol=1)
    ax.text(
        1.02, 0.42,
        "Error bars: stored training-CV macro-F1 SD only\nBest indicated by saved Phase 5 validation ranking",
        transform=ax.transAxes, ha="left", va="top", fontsize=8, color="#444444",
    )
    fig.tight_layout(rect=(0, 0, 0.82, 1))
    outputs = save_figure(fig, "figure_5_model_comparison", EXPORT_FORMATS)
    outputs.append(summary_path)
    best_name = canonical[selected_models.iloc[0]["model"]]
    record(5, "Machine-learning model comparison", sources, outputs, "COMPLETE",
           f"{best_name} is the Phase 5 validation-selected model (saved rank 1).")
except Exception as exc:
    record(5, "Machine-learning model comparison", sources, outputs, "SKIPPED", str(exc))

Figure 5: COMPLETE — Machine-learning model comparison
  XGBoost is the Phase 5 validation-selected model (saved rank 1).


## Figure 6 — Confusion matrix for the best classifier

Phase 5 exported aggregate comparison metrics and a per-class report, but it
did **not** export held-out true labels or predictions. Those arrays exist only
as transient variables in Notebook 05. Recreating them would require
regenerating the split and rerunning a trained model, both explicitly
prohibited for Phase 9. Therefore this figure is intentionally skipped.

In [8]:
prediction_candidates = [
    PROJECT / "results" / "tables" / "classification_test_predictions.csv",
    PROJECT / "results" / "tables" / "classification_heldout_predictions.csv",
    PROJECT / "results" / "tables" / "classification_predictions.csv",
]
available_prediction_files = [p for p in prediction_candidates if p.is_file()]
if not available_prediction_files:
    record(
        6, "Confusion matrix for the best classifier",
        [SOURCE["models"], *prediction_candidates], [], "SKIPPED",
        "Phase 5 did not save held-out true labels and predictions. Regenerating the split "
        "or retraining/rerunning a classifier is prohibited, so counts and normalized "
        "confusion matrices cannot be produced without fabrication.",
    )
else:
    record(
        6, "Confusion matrix for the best classifier",
        available_prediction_files, [], "SKIPPED",
        "A candidate prediction file exists but no validated Phase 5 prediction schema was "
        "documented in the existing notebook; no values were guessed.",
    )

Figure 6: SKIPPED — Confusion matrix for the best classifier
  Phase 5 did not save held-out true labels and predictions. Regenerating the split or retraining/rerunning a classifier is prohibited, so counts and normalized confusion matrices cannot be produced without fabrication.


## Figure 7 — Phase 8 biological reasoning summary

Source: `results/phase8/all_clusters_summary.csv`. Qualitative categories are
shown as ordered labeled positions, not converted to probabilities.

In [9]:
sources = [SOURCE["phase8_summary"]]
outputs = []
try:
    require_files(sources)
    required = {
        "Cluster ID", "Proposed Cell Type", "Annotation Support", "Overall Confidence",
        "Dominant Biological Program", "Validation Status",
    }
    if phase8 is None or not required.issubset(phase8.columns):
        raise ValueError(f"Phase 8 summary is missing columns: {sorted(required - set(phase8.columns if phase8 is not None else []))}")
    summary = phase8.sort_values("Cluster ID", key=cluster_sort_key).copy()
    summary_path = OUTPUT_DIR / "biological_reasoning_summary.csv"
    summary.to_csv(summary_path, index=False)

    support_order = ["weakly supported", "partially supported", "moderately supported", "strongly supported"]
    confidence_order = ["Low", "Moderate", "High"]
    support_x = {value: i for i, value in enumerate(support_order)}
    confidence_x = {value: i for i, value in enumerate(confidence_order)}
    unknown_support = sorted(set(summary["Annotation Support"]) - set(support_x))
    unknown_confidence = sorted(set(summary["Overall Confidence"]) - set(confidence_x))
    if unknown_support or unknown_confidence:
        raise ValueError(f"Unexpected qualitative categories: support={unknown_support}, confidence={unknown_confidence}")

    y = np.arange(len(summary))
    fig = plt.figure(figsize=(16, 8))
    grid = fig.add_gridspec(1, 3, width_ratios=[1.15, 0.95, 2.9], wspace=0.18)
    ax1 = fig.add_subplot(grid[0, 0])
    ax2 = fig.add_subplot(grid[0, 1])
    ax3 = fig.add_subplot(grid[0, 2])
    colors = [CELL_TYPE_COLORS[x] for x in summary["Proposed Cell Type"]]

    ax1.scatter([support_x[x] for x in summary["Annotation Support"]], y, s=130, c=colors, edgecolor="white")
    ax1.set(
        title="Annotation support", xlabel="Ordered qualitative category",
        xticks=range(len(support_order)), xticklabels=["Weak", "Partial", "Moderate", "Strong"],
        yticks=y,
        yticklabels=[f"C{r[0]} · {r[1]}" for r in summary.itertuples(index=False, name=None)],
    )
    ax1.tick_params(axis="x", rotation=30)
    ax1.invert_yaxis()
    ax1.grid(axis="y", visible=False)

    confidence_colors = {"Low": "#D55E00", "Moderate": "#E69F00", "High": "#009E73"}
    ax2.scatter(
        [confidence_x[x] for x in summary["Overall Confidence"]], y, s=130,
        c=[confidence_colors[x] for x in summary["Overall Confidence"]], edgecolor="white",
    )
    ax2.set(
        title="Overall confidence", xlabel="Ordered qualitative category",
        xticks=range(len(confidence_order)), xticklabels=confidence_order,
    )
    ax2.set_ylim(len(summary) - 0.5, -0.5)
    ax2.set_yticks([])
    ax2.tick_params(axis="x", rotation=30)
    ax2.grid(axis="y", visible=False)

    ax3.set_xlim(0, 1)
    ax3.set_ylim(len(summary) - 0.5, -0.5)
    ax3.axis("off")
    ax3.set_title("Dominant biological program and validation", loc="left")
    for i, row in enumerate(summary.itertuples(index=False, name=None)):
        program = textwrap.fill(row[4], width=48)
        status = row[5]
        ax3.text(0.0, i, program, va="center", fontsize=9)
        ax3.text(
            0.98, i, f"✓ {status}", ha="right", va="center", fontsize=9,
            color="#007A4D", fontweight="bold",
            bbox={"boxstyle": "round,pad=0.25", "facecolor": "#E7F6EF", "edgecolor": "#8AC8AA"},
        )
    fig.suptitle("Phase 8 evidence-grounded biological reasoning summary", fontsize=16, y=0.98)
    outputs = save_figure(fig, "figure_7_biological_reasoning_summary", ("png",))
    outputs.append(summary_path)
    record(7, "Phase 8 biological reasoning summary", sources, outputs, "COMPLETE",
           "Qualitative support and confidence retain their validated category labels; all status labels are displayed.")
except Exception as exc:
    record(7, "Phase 8 biological reasoning summary", sources, outputs, "SKIPPED", str(exc))

/var/folders/z1/f02bmsc17gd00g8pl4m2tnxc0000gn/T/ipykernel_62420/2606454080.py:100: UserWarning: Glyph 10003 (\N{CHECK MARK}) missing from font(s) Arial.
  fig.savefig(path, **save_kwargs)


Figure 7: COMPLETE — Phase 8 biological reasoning summary
  Qualitative support and confidence retain their validated category labels; all status labels are displayed.


## Figure 8 — Evidence and validation overview

Sources: Phase 6 selected markers, Phase 7 literature/references/coverage/reuse
and validation artifacts, and the Phase 8 all-cluster summary. All totals are
calculated from current files and compared with the expected project totals.

In [10]:
sources = [
    SOURCE["markers"], SOURCE["phase7_literature"], SOURCE["phase7_references"],
    SOURCE["phase7_coverage"], SOURCE["phase7_reuse"], SOURCE["phase7_validation"],
    SOURCE["phase8_summary"],
]
outputs = []
try:
    require_files(sources)
    literature = pd.read_csv(SOURCE["phase7_literature"])
    references = pd.read_csv(SOURCE["phase7_references"], dtype={"PMID": str})
    coverage = pd.read_csv(SOURCE["phase7_coverage"])
    reuse = pd.read_csv(SOURCE["phase7_reuse"])
    validation = json.loads(SOURCE["phase7_validation"].read_text(encoding="utf-8"))
    phase8_current = pd.read_csv(SOURCE["phase8_summary"])

    totals = [
        ("Clusters", int(markers["cluster"].nunique()), 9),
        ("Cluster–gene entries", int(len(markers)), 90),
        ("Unique genes reviewed", int(markers["gene"].nunique()), 78),
        ("Verified reference rows", int(len(references)), 231),
        ("Unique verified PMIDs", int(references["PMID"].dropna().nunique()), 224),
        ("Reused genes", int(reuse["evidence_reused"].fillna(False).astype(bool).sum()), 11),
        ("Phase 8 validation passes", int(phase8_current["Validation Status"].eq("PASS").sum()), 9),
        ("Phase 8 validation failures", int(phase8_current["Validation Status"].eq("FAIL").sum()), 0),
        ("Insufficient-evidence genes", int(coverage["genes_with_insufficient_evidence"].sum()), 1),
    ]
    overview = pd.DataFrame(totals, columns=["metric", "calculated_value", "expected_value"])
    overview["matches_expected"] = overview["calculated_value"].eq(overview["expected_value"])

    fig, ax = plt.subplots(figsize=(15, 8))
    ax.axis("off")
    ax.set_xlim(0, 3)
    ax.set_ylim(0, 3.5)
    card_colors = ["#E8F1FA", "#EDF7F2", "#FFF4DF", "#F3EEFA"]
    for i, row in enumerate(overview.itertuples(index=False)):
        col, grid_row = i % 3, i // 3
        x, y0 = col + 0.08, 2.45 - grid_row * 1.05
        card = FancyBboxPatch(
            (x, y0), 0.84, 0.82, boxstyle="round,pad=0.03,rounding_size=0.04",
            facecolor=card_colors[i % len(card_colors)],
            edgecolor="#C8D0D8", linewidth=1.0,
        )
        ax.add_patch(card)
        ax.text(x + 0.42, y0 + 0.52, f"{row.calculated_value:,}", ha="center", va="center",
                fontsize=24, fontweight="bold", color="#1C2B39")
        ax.text(x + 0.42, y0 + 0.22, textwrap.fill(row.metric, 24), ha="center", va="center",
                fontsize=9, color="#334E68")
        if not row.matches_expected:
            ax.text(x + 0.42, y0 + 0.05, f"Expected {row.expected_value:,}", ha="center",
                    va="bottom", fontsize=7, color="#B00020")
    ax.text(0.08, 3.38, "Evidence and validation overview", fontsize=18, fontweight="bold", va="top")
    subtitle = (
        "Calculated directly from current Phase 6–8 artifacts"
        if overview["matches_expected"].all()
        else "Calculated totals shown; red notes identify discrepancies from expected project totals"
    )
    ax.text(0.08, 3.16, subtitle, fontsize=10, color="#52616B", va="top")
    ax.text(
        2.92, 3.38, f"Phase 7 validation: {validation.get('status', 'UNKNOWN')}",
        ha="right", va="top", fontsize=11, fontweight="bold",
        color="#007A4D" if validation.get("status") == "PASS" else "#B00020",
    )
    outputs = save_figure(fig, "figure_8_evidence_validation_overview", ("png",))
    mismatch = overview.loc[~overview["matches_expected"], "metric"].tolist()
    record(8, "Evidence and validation overview", sources, outputs, "COMPLETE",
           "All calculated totals match the expected project totals." if not mismatch
           else "Calculated totals differ from expected for: " + ", ".join(mismatch))
except Exception as exc:
    record(8, "Evidence and validation overview", sources, outputs, "SKIPPED", str(exc))

Figure 8: COMPLETE — Evidence and validation overview
  All calculated totals match the expected project totals.


## Figure 9 — Complete project pipeline diagram

Sources: existing Phase 1–8 notebooks and validated output directories. The
diagram communicates provenance and separates computational analysis,
literature integration, evidence-grounded reasoning, and validation.

In [11]:
notebook_sources = [PROJECT / "notebooks" / f"{i:02d}_{name}.ipynb" for i, name in [
    (1, "explore_all_pbmc3k_files"),
    (2, "qc_preprocessing"),
    (3, "eda_clustering"),
    (4, "marker_gene_discovery"),
    (5, "classification_model_comparison"),
    (6, "cluster_biological_interpretation"),
    (7, "literature_integration"),
    (8, "evidence_grounded_biological_reasoning"),
]]
sources = notebook_sources + [SOURCE["phase8_summary"]]
outputs = []
try:
    require_files(sources)
    lanes = [
        ("Input", "#ECEFF4", ["PBMC3K\nraw data"]),
        ("Computational analysis", "#DCEAF7", [
            "Explore", "Quality\ncontrol", "Preprocess", "Dimension\nreduction",
            "Cluster", "Cell\nannotation", "Marker\ndiscovery", "ML\nclassification",
        ]),
        ("Literature integration", "#FFF0D6", ["Verified literature\nintegration"]),
        ("Evidence-grounded reasoning", "#EAE1F5", ["Cluster-level\nbiological reasoning"]),
        ("Validation", "#DFF2E6", ["Schema · gene · citation\nconfidence · safety checks"]),
        ("Output", "#D7EEE4", ["Validated reports\nfor all 9 clusters"]),
    ]
    steps = []
    for lane, color, labels in lanes:
        for label in labels:
            steps.append((lane, color, label))

    fig, ax = plt.subplots(figsize=(20, 7))
    ax.axis("off")
    ax.set_xlim(0, len(steps) + 0.7)
    ax.set_ylim(-0.4, 2.35)
    y = 0.65
    for i, (lane, color, label) in enumerate(steps):
        x = i + 0.18
        box = FancyBboxPatch(
            (x, y), 0.78, 0.62, boxstyle="round,pad=0.035,rounding_size=0.05",
            facecolor=color, edgecolor="#52616B", linewidth=1.0,
        )
        ax.add_patch(box)
        ax.text(x + 0.39, y + 0.31, label, ha="center", va="center", fontsize=8.5, fontweight="bold")
        if i < len(steps) - 1:
            ax.annotate(
                "", xy=(x + 1.02, y + 0.31), xytext=(x + 0.82, y + 0.31),
                arrowprops={"arrowstyle": "-|>", "lw": 1.2, "color": "#52616B"},
            )

    start = 0
    for lane, color, labels in lanes:
        width = len(labels)
        x0 = start + 0.16
        x1 = start + width - 0.02
        ax.plot([x0, x1], [1.6, 1.6], color="#8795A1", linewidth=1)
        ax.plot([x0, x0], [1.53, 1.67], color="#8795A1", linewidth=1)
        ax.plot([x1, x1], [1.53, 1.67], color="#8795A1", linewidth=1)
        ax.text((x0 + x1) / 2, 1.78, lane, ha="center", va="center", fontsize=9,
                fontweight="bold", color="#334E68")
        start += width
    ax.text(
        0.18, 2.22, "PBMC3K complete validated analysis pipeline",
        fontsize=19, fontweight="bold", va="top",
    )
    ax.text(
        0.18, 1.99,
        "Preprocessing, clustering, model training, and literature verification are upstream computational steps; "
        "Phase 8 consumes their validated outputs for evidence-grounded reasoning.",
        fontsize=9.5, color="#52616B", va="top",
    )
    fig.tight_layout()
    outputs = save_figure(fig, "figure_9_complete_pipeline", EXPORT_FORMATS)
    record(9, "Complete project pipeline diagram", sources, outputs, "COMPLETE",
           "Computational analysis, literature integration, reasoning, and validation are explicitly separated.")
except Exception as exc:
    record(9, "Complete project pipeline diagram", sources, outputs, "SKIPPED", str(exc))

Figure 9: COMPLETE — Complete project pipeline diagram
  Computational analysis, literature integration, reasoning, and validation are explicitly separated.


## Figure 10 — Cluster summary table

Sources: Phase 6 representative markers and the Phase 8 all-cluster reasoning
summary.

In [12]:
sources = [SOURCE["markers"], SOURCE["phase8_summary"]]
outputs = []
try:
    require_files(sources)
    top_markers = (
        markers.sort_values(["cluster", "representative_rank"], key=lambda s: cluster_sort_key(s) if s.name == "cluster" else s)
        .groupby("cluster", observed=True)["gene"]
        .apply(lambda x: ", ".join(x.head(5).astype(str)))
        .rename("Top representative markers")
        .reset_index().rename(columns={"cluster": "Cluster ID"})
    )
    summary = phase8.merge(top_markers, on="Cluster ID", how="left", validate="one_to_one")
    final = summary[[
        "Cluster ID", "Proposed Cell Type", "Top representative markers",
        "Annotation Support", "Overall Confidence", "Dominant Biological Program",
        "Validation Status",
    ]].sort_values("Cluster ID", key=cluster_sort_key)
    final_path = OUTPUT_DIR / "final_cluster_summary.csv"
    final.to_csv(final_path, index=False)

    display_table = final.copy()
    display_table["Cluster ID"] = "C" + display_table["Cluster ID"].astype(str)
    display_table["Proposed Cell Type"] = display_table["Proposed Cell Type"].map(lambda x: textwrap.fill(x, 24))
    display_table["Top representative markers"] = display_table["Top representative markers"].map(lambda x: textwrap.fill(x, 28))
    display_table["Annotation Support"] = display_table["Annotation Support"].map(lambda x: textwrap.fill(x, 18))
    display_table["Dominant Biological Program"] = display_table["Dominant Biological Program"].map(lambda x: textwrap.fill(x, 32))
    display_table["Validation Status"] = display_table["Validation Status"].map(lambda x: "✓ " + x)
    display_table = display_table.rename(columns={
        "Cluster ID": "Cluster",
        "Proposed Cell Type": "Final cell type",
        "Top representative markers": "Top Phase 6 markers",
        "Annotation Support": "Support",
        "Overall Confidence": "Confidence",
        "Dominant Biological Program": "Dominant program",
        "Validation Status": "Validation",
    })

    fig, ax = plt.subplots(figsize=(20, 8.7))
    ax.axis("off")
    table = ax.table(
        cellText=display_table.values,
        colLabels=display_table.columns,
        cellLoc="left", colLoc="left",
        colWidths=[0.055, 0.17, 0.20, 0.12, 0.085, 0.27, 0.10],
        loc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(8.2)
    table.scale(1, 2.55)
    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor("#D5DADF")
        cell.set_linewidth(0.7)
        if row == 0:
            cell.set_facecolor("#243B53")
            cell.get_text().set_color("white")
            cell.get_text().set_fontweight("bold")
        else:
            cell.set_facecolor("#F7F9FB" if row % 2 == 0 else "white")
            if col == 0:
                cell.get_text().set_fontweight("bold")
                cell.get_text().set_color(CELL_TYPE_COLORS[final.iloc[row - 1]["Proposed Cell Type"]])
            if col == 6:
                cell.get_text().set_color("#007A4D")
                cell.get_text().set_fontweight("bold")
    ax.set_title("PBMC3K final cluster summary", fontsize=18, fontweight="bold", loc="left", pad=18)
    outputs = save_figure(fig, "figure_10_cluster_summary_table", ("png", "pdf"))
    outputs.append(final_path)
    record(10, "Cluster summary table", sources, outputs, "COMPLETE",
           "Top five Phase 6 representative markers are shown for each validated Phase 8 cluster.")
except Exception as exc:
    record(10, "Cluster summary table", sources, outputs, "SKIPPED", str(exc))

/var/folders/z1/f02bmsc17gd00g8pl4m2tnxc0000gn/T/ipykernel_62420/2606454080.py:100: UserWarning: Glyph 10003 (\N{CHECK MARK}) missing from font(s) Arial.
  fig.savefig(path, **save_kwargs)


Figure 10: COMPLETE — Cluster summary table
  Top five Phase 6 representative markers are shown for each validated Phase 8 cluster.


## Manifest and final report

In [13]:
# Always write the manifest and summary, including skipped or failed figures.
manifest = pd.DataFrame(manifest_rows).sort_values("figure_number")
manifest_path = OUTPUT_DIR / "figure_manifest.csv"
manifest.to_csv(manifest_path, index=False)

complete = manifest.loc[manifest["status"].isin(["COMPLETE", "PARTIAL"])]
skipped = manifest.loc[manifest["status"].isin(["SKIPPED", "FAILED"])]

def markdown_items(values):
    values = list(values)
    return "\n".join(f"- {value}" for value in values) if values else "- None"

finding_lines = []
if (OUTPUT_DIR / "cluster_composition.csv").is_file():
    comp = pd.read_csv(OUTPUT_DIR / "cluster_composition.csv")
    largest = comp.sort_values("cell_count", ascending=False).iloc[0]
    smallest = comp.sort_values("cell_count").iloc[0]
    finding_lines.append(
        f"The largest cluster is C{largest.cluster_id} ({largest.cell_type}; "
        f"{int(largest.cell_count):,} cells, {largest.percentage_of_total:.1f}%)."
    )
    finding_lines.append(
        f"The smallest cluster is C{smallest.cluster_id} ({smallest.cell_type}; "
        f"{int(smallest.cell_count):,} cells, {smallest.percentage_of_total:.1f}%)."
    )
if (OUTPUT_DIR / "model_comparison_summary.csv").is_file():
    model_summary = pd.read_csv(OUTPUT_DIR / "model_comparison_summary.csv")
    selected_name = model_summary.loc[model_summary["is_phase5_selected_best"], "model"].iloc[0]
    finding_lines.append(
        f"{selected_name} remains the Phase 5 validation-selected classifier; Phase 9 does not reinterpret "
        "the selection using held-out test metrics."
    )
if phase8 is not None:
    finding_lines.append(
        f"Phase 8 reports {int(phase8['Validation Status'].eq('PASS').sum())} validation passes and "
        f"{int(phase8['Validation Status'].eq('FAIL').sum())} failures."
    )

summary_path = OUTPUT_DIR / "visualization_summary.md"
all_output_paths = sorted(
    [p for p in OUTPUT_DIR.iterdir() if p.is_file() and p != summary_path]
    + [summary_path]
)
summary_text = f"""# Phase 9 Visualization Summary

## Execution status

- Figures complete or partial: **{len(complete)}**
- Figures skipped or failed: **{len(skipped)}**
- Manifest: `{rel(manifest_path)}`

## Figures successfully generated

{markdown_items(f"Figure {r.figure_number}: {r.figure_name} ({r.status})" for r in complete.itertuples())}

## Figures skipped or failed

{markdown_items(f"Figure {r.figure_number}: {r.figure_name} — {r.notes}" for r in skipped.itertuples())}

## Source files used

{markdown_items(sorted(set(s for cell in manifest["source_files"] for s in str(cell).split("; ") if s and "classification_" not in s or Path(PROJECT / s).is_file())))}

## Key visual findings

{markdown_items(finding_lines)}

## Missing-data limitations

{markdown_items(r.notes for r in skipped.itertuples())}

## Output paths

{markdown_items(f"`{rel(p)}`" for p in all_output_paths)}
"""
summary_path.write_text(summary_text, encoding="utf-8")

display(manifest)
print(f"Saved manifest: {rel(manifest_path)}")
print(f"Saved summary: {rel(summary_path)}")
print(f"Phase 9 files: {len(list(OUTPUT_DIR.iterdir()))}")

,figure_number,figure_name,source_files,output_files,status,notes
0,1,Final annotated UMAP,data/processed/pbmc3k_phase3_annotated.h5ad; r...,,SKIPPED,unhashable type: 'list'
1,2,Cluster and cell-type composition,data/processed/pbmc3k_phase3_annotated.h5ad; r...,results/phase9/figure_2_cluster_cell_counts.pn...,COMPLETE,"Counts and percentages sum to 2,638 cells and ..."
2,3,Representative marker-gene heatmap,data/processed/pbmc3k_phase3_annotated.h5ad; r...,results/phase9/figure_3_marker_gene_heatmap.pn...,COMPLETE,Plotted 78 deduplicated Phase 6 genes as gene-...
3,4,Marker specificity summary,results/phase6/selected_marker_genes.csv,results/phase9/figure_4_marker_specificity.png...,COMPLETE,Top five saved representative markers per clus...
4,5,Machine-learning model comparison,results/tables/classification_model_comparison...,results/phase9/figure_5_model_comparison.png; ...,COMPLETE,XGBoost is the Phase 5 validation-selected mod...
5,6,Confusion matrix for the best classifier,results/tables/classification_model_comparison...,,SKIPPED,Phase 5 did not save held-out true labels and ...
6,7,Phase 8 biological reasoning summary,results/phase8/all_clusters_summary.csv,results/phase9/figure_7_biological_reasoning_s...,COMPLETE,Qualitative support and confidence retain thei...
7,8,Evidence and validation overview,results/phase6/selected_marker_genes.csv; resu...,results/phase9/figure_8_evidence_validation_ov...,COMPLETE,All calculated totals match the expected proje...
8,9,Complete project pipeline diagram,notebooks/01_explore_all_pbmc3k_files.ipynb; n...,results/phase9/figure_9_complete_pipeline.png;...,COMPLETE,"Computational analysis, literature integration..."
9,10,Cluster summary table,results/phase6/selected_marker_genes.csv; resu...,results/phase9/figure_10_cluster_summary_table...,COMPLETE,Top five Phase 6 representative markers are sh...


Saved manifest: results/phase9/figure_manifest.csv
Saved summary: results/phase9/visualization_summary.md
Phase 9 files: 28


## Reproducibility note

The notebook uses relative project discovery, fixed plotting configuration, and
a fixed random seed. No scientific outputs in Phases 1–8 are written or
modified. Derived Phase 9 data are saved only beneath `results/phase9/`.